# Analyze: Multiple Scenario PartMC vs. MAM4

This notebook demonstrates how to analyze results from a multiple-scenario AMBRS comparison of PartMC and MAM4. You will use a variety of diagnostic plots to understand differences in structural errors between the PartMC single-particle model and the MAM4 modal model.

By the end of this tutorial, you should be able to:
* Understand AMBRS "cases" and how they are applied to various aerosol models
* Plot diagnostic properties from AMBRS output data
* Use diagnostics to understand the effects of structural error in PartMC and MAM4
* Apply this understanding of structural error effects to predict differences in additional cases

<small>__Note:__ This notebook expects AMBRS output data for the multiple-scenario comparison of PartMC and MAM4 to exist in `tutorial/data/prepared/multiple_scenario_partmc/`. The output data can be regenerated using `tutorial/notebooks/01_run_models/02_multiple_scenario_partmc.ipynb`.</small>

## About our Models

__PartMC:__ Single-particle aerosol representation. Particles treated as well-mixed liquid spheres.

__MAM4:__ Modal aerosol representation. 4 log-normal modes with fixed geometric standard deviation and variable number concentration and geometric mean diameter. Particles treated as well-mixed liquid spheres.

## Setup

Here we'll pull in AMBRS dependencies and create a few helpers so that it's easy to navigate the AMBRS data.

__Note:__ If you are running this notebook in Google Colab, you will get a pop-up after running the following cell for the first time saying that you need to restart the session. Click the `restart session` button and re-run the cell. It should work the second time.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    repo_dir = "/content/ambrs"
    setup_marker = Path("/content/.ambrs_setup_complete")

    if not setup_marker.exists():
        %pip uninstall -y splot pymc tsfresh pysal giddy
        %pip uninstall -y numba librosa pytensor segregation umap-learn pynndescent quantecon stumpy
        %pip install numpy==2.5.3
        if not os.path.isdir(repo_dir):
            subprocess.run(
                [
                    "git", "clone", "--depth", "1", "--branch", "develop-148-tutorial-notebook",
                    "https://github.com/cohere-llc/ambrs.git",
                    repo_dir,
                ],
                check=True,
            )

        subprocess.run(
            [
                sys.executable, "-m", "pip", "install",
                "-e", f"{repo_dir}[colab]",
            ],
            check=True,
        )
        setup_marker.touch()

    os.chdir(f"{repo_dir}/tutorial/notebooks/02_plot_results")
    print("Colab setup complete.")
else:
    print("Not running in Colab; using the current environment.")

In [ ]:
import pickle
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display
from part2pop.viz.builder import build_plotter
from part2pop.viz.style import StyleManager, Theme

tutorial_root = Path.cwd().parent.parent

# post-processed diagnostic data will be saved here
prepared_output_dir = (
    tutorial_root
    / 'data'
    / 'prepared'
    / 'multiple_scenario_partmc_mam4'
)

model_dropdown = widgets.Dropdown(
    options=[
        '',
        'PartMC',
        'MAM4',
        'PartMC and MAM4',
    ],
    value='',
    description='Select model(s):',
    disabled=False,
)

process_dropdown = widgets.Dropdown(
    options=[
        '',
        'coagulation only',
        'condensation only',
        'condensation with coagulation',
    ],
    value='',
    description='Select processes:',
    disabled=False,
)

scenario_dropdown = widgets.Dropdown(
    options=[
        '',
        'base case',
        'high particle count',
        'high sulfuric acid',
    ],
    value='',
    description='Select a scenario:',
    disabled=False,
)

diagnostic_dropdown = widgets.Dropdown(
    options=[
        '',
        'b_scat',
        'dNdlnD',
        'frac_ccn',
        'geom_mean_diameter',
        'geom_std_dev',
        'number_concentration',
    ],
    value='',
    description='Select a diagnostic:',
    disabled=False,
)

is_snapshot = ['b_scat', 'dNdlnD', 'frac_ccn']
timestep_dropdown = widgets.Dropdown(
    options=[
        '0',
        '12', 
        '24',
        '48',
        '60',
    ],
    value='0',
    description='Select a timestep (min):',
    disabled=False,
)

for dropdown in (
    model_dropdown,
    process_dropdown,
    scenario_dropdown,
    diagnostic_dropdown,
    timestep_dropdown,
):
    dropdown.style.description_width = '160px'
    dropdown.layout.width = '450px'

def plot():
    if '' in [model_dropdown.value, process_dropdown.value, scenario_dropdown.value, diagnostic_dropdown.value]:
        return
    suffix = (f'-{timestep_dropdown.value}' if diagnostic_dropdown.value in is_snapshot else '')
    pkl_file = f'{process_dropdown.value}-{scenario_dropdown.value}-{diagnostic_dropdown.value}{suffix}.pkl'.replace(' ', '-')
    with (prepared_output_dir / pkl_file).open('rb') as f:
        payload = pickle.load(f)

    mgr = StyleManager(Theme(), deterministic=False)
    plotter = build_plotter(
        payload['plotter_type'],
        payload['plotter_config'],
    )

    series = [
        *([{"key": "partmc", "population": payload['prepared']['partmc'], "label": "PartMC-MOSAIC"}] if 'PartMC' in model_dropdown.value else []),
        *([{"key": "mam4", "population": payload['prepared']['mam4'], "label": "MAM4"}] if 'MAM4' in model_dropdown.value else []),
    ]
    line_styles = mgr.plan("line", [s["key"] for s in series])

    fig, ax = plt.subplots()
    for s in series:
        cfg = {
            "varname": diagnostic_dropdown.value,
            "style": line_styles[s["key"]],
        }
        plotter = build_plotter(payload['plotter_type'], cfg)
        plotter.plot(ax=ax, prepared=s['population'], label=s["label"])
    time_text = (f'; time = {payload["metadata"]["time"]} s' if 'time' in payload['metadata'] else '')
    fig.text(0, 0, f'{process_dropdown.value} - {scenario_dropdown.value}{time_text}')
    ax.legend()
    if diagnostic_dropdown.value == 'number_concentration':
        if scenario_dropdown.value == "high particle count":
            ax.set_ylim(bottom=0.0, top=5.5e14)
        else:
            ax.set_ylim(bottom=0.0, top=5.5e12)
    if diagnostic_dropdown.value == 'dNdlnD':
        ax.set_xlim(right=1e-5)
    fig.tight_layout()

def update_timestep_visibility(change=None):
    timestep_dropdown.layout.display = (
        '' if diagnostic_dropdown.value in is_snapshot else 'none'
    )
diagnostic_dropdown.observe(update_timestep_visibility, names='value')
update_timestep_visibility()

def show_dropdowns():
    display(model_dropdown)
    display(scenario_dropdown)
    display(process_dropdown)
    display(diagnostic_dropdown)
    display(timestep_dropdown)


## AMBRS Cases

In AMBRS, "cases" are used to create reproducible model comparisons. Cases define all AMBRS options including environmental conditions, initial gas and particle states, model configurations, and diagnostic calculations. Cases are made up of three components: scenarios, model configuration, and diagnostics.

### Scenarios

Scenarios define the initial and evolving conditions of a simulation. These include time-resolved temperature, pressure, and air density, the initial concentrations of gas-phase species, the intial particle population, and emissions of gases and aerosols. Using consistent conditions (and multiple cases) facilitates investigation of differences in structural errors among models.

For this tutorial, we'll use three scenarios. All three scenarios start with a single log-normal distribution of aerosol particles that contain a mixture of sulfate and non-volatile organic matter. In the gas-phase we have only sulfuric acid in dry conditions. The three scenarios differ only in the inital particle count and gas-phase sulfuric acid concentration.

| Label | H2SO4(g) [mol mol-1] | Particle Count [# m-3] | SO4(a) [mass fraction, kg kg-1] | POM(a) [mass fraction, kg kg-1] |
|-----|-----|-----|-----|-----|
| Base Case | 1.0e-5 | 5.0e12 | 0.45 | 0.55 |
| High Particle Count | 1.0e-5 | 5.0e14 | 0.45 | 0.55 |
| High Sulfuric Acid | 1.0e-3 | 5.0e12 | 0.45 | 0.55 |

In some cases, the four modes used by MAM4 to represent an aerosol system limit its ability to accurately capture certain aerosol populations, leading to differences in the initial size distribution compared to the higher-resolution PartMC model. For this scenario, we choose our initial aerosol popultion to correspond to the "accumulation" mode in MAM4. Our initial conditions should therefor be similar in both PartMC and MAM4 aerosol representations.

_What types of aerosol systems would PartMC's assumptions prevent it from accurately capturing?_

### Model Configuration

PartMC and MAM4 can be configured to treat many physical and chemical processes during a simulation. AMBRS is designed to allow you to choose consistent configuration options for the models you are comparing. Let's look at the code we used to set up these tutorial cases:

```python
# simulation timing
dt = 60  # time step (s)
nstep = 60  # number of time steps (#) = 1 hr

# physical processes to include in the simulations
model_configs = {
    'condensation-only': ambrs.aerosol.AerosolProcesses(
        coagulation=False,
        condensation=True,
    ),
    'coagulation-only': ambrs.aerosol.AerosolProcesses(
        coagulation=True,
        condensation=False,
    ),
    'condensation-with-coagulation': ambrs.aerosol.AerosolProcesses(
        coagulation=True,
        condensation=True,
    ),
}
```
We use the same number and length of timesteps for both models, and use the `AerosolProcesses` objects to define three configurations that can be applied to both PartMC and MAM4:

* condensation only
* coagulation only
* condensation with coagulation

### Diagnostics

The final component of an AMBRS case are the diagnostic calcluation configurations. AMBRS uses the [`part2pop`](https://github.com/pnnl/part2pop) package for diagnostic calcluations and plotting. For this tutorial, we'll include six diagnostics: three time-series and three snapshots.

__Time Series__
* Particle Number Concentration
* Geometric Mean Diameter
* Geometric Standard Deviation

__Snapshots__
* dN/dlog(Dp) vs. log(Dp)
* CCN Number vs. % Supersaturation
* Scattering Coefficient vs. Wavelength

With three possible scenarios and three possible model configurations, we have a total of nine cases to examine:

* condensation only: base case
* condensation only: high particle count
* condensation only: high sulfuric acid
* coagulation only: base case
* coagulation only: high particle count
* coagulation only: high sulfuric acid
* condensation with coagulation: base case
* condensation with coagulation: high particle count
* condensation with coagulation: high sulfuric acid

## Predict the Differences

Now, let's look at comparisons of full simulations using PartMC and MAM4. First, choose a case, and then choose a diagnostic to plot (for snapshot diagnostics, you will also need to select a timestep to plot):

In [ ]:
show_dropdowns()

_Based on what you know of how PartMC and MAM4 represent aerosol systems, what differences would you expect to see in this property over time?_

## Plot the Results

Let's see if you were right! Plot the chosen diagnostic:

In [ ]:
plot()